## Colab setup (run this first)

Mounts Google Drive, moves into the project folder, and installs the GPU
packages. Run this once per session, then **Runtime -> Restart session** if
prompted after the install, and continue from Section 0.

If you are **not** on Colab (e.g. running locally), skip this cell.

In [1]:
# ============================================================
# COLAB SETUP - run first (safe to skip when running locally)
# ============================================================
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    print("[setup] Not on Colab -> skipping Drive mount & installs.")

if IN_COLAB:
    # 1) Mount Google Drive so the project files are visible.
    from google.colab import drive
    drive.mount('/content/drive')

    # 2) Move into the project folder (adjust if yours lives elsewhere).
    import os
    PROJECT_DIR = '/content/drive/MyDrive/MiniProject'
    assert os.path.isdir(PROJECT_DIR), (
        f"PROJECT_DIR not found: {PROJECT_DIR}\n"
        "Edit PROJECT_DIR to match where you uploaded the MiniProject folder. "
        "Tip: print(os.listdir('/content/drive/MyDrive')) to find it."
    )
    os.chdir(PROJECT_DIR)
    print("[setup] Now in:", os.getcwd())
    print("[setup] Files here:", os.listdir())

    # 3) Install pinned GPU packages for Module 3 (classical stack is preinstalled).
    #    transformers 4.44.2 keeps the TrainingArguments API the notebook expects.
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==4.44.2", "datasets", "accelerate",
                    "scikit-learn", "networkx"], check=False)
    print("[setup] Installs done. If Colab asks, Runtime -> Restart session, "
          "then re-run from here.")

    # 4) Report GPU status (Module 3 trains only when a GPU is visible).
    try:
        import torch
        print("[setup] CUDA available:", torch.cuda.is_available())
        if torch.cuda.is_available():
            print("[setup] GPU:", torch.cuda.get_device_name(0))
    except Exception as _e:
        print("[setup] torch not importable yet:", _e)


Mounted at /content/drive
[setup] Now in: /content/drive/MyDrive/MiniProject
[setup] Files here: ['SMS PHISHING DATASET FOR MACHINE LEARNING AND PATTERN RECOGNITION.zip', 'Dataset_5971.csv', 'common.py', '02_cuml_baseline.py', '03_nemo_bert_training.py', '04_cugraph_analysis.py', '01_data_prep.py', 'README.md', 'outputs', '__pycache__', 'SMS PHISHING DATASET FOR MACHINE LEARNING AND PATTERN RECOGNITION', 'artifacts', 'ScamDetection_Pipeline.ipynb']
[setup] Installs done. If Colab asks, Runtime -> Restart session, then re-run from here.
[setup] CUDA available: True
[setup] GPU: Tesla T4


# Scam Message Detection & Graph Analytics — End-to-End GPU Pipeline

NVIDIA RAPIDS / NeMo lab patterns: **cuDF, cuML, NeMo + PyTorch Lightning, cuGraph / nx-cugraph**, each with a CPU fallback so the notebook runs anywhere.

**Modules**
1. GPU data preprocessing → NeMo-format TSV splits (70/15/15 stratified)
2. cuML TF-IDF + LogisticRegression baseline
3. NeMo BERT fine-tuning (cased-vs-uncased, seq-length sweep) — **GPU only**
4. Word co-occurrence graph: PageRank + Louvain

Run the cells top to bottom. Module 1 must run before Modules 2–4 (it writes the TSVs they read). Module 3 needs an NVIDIA GPU with NeMo installed; on CPU it prints setup guidance and skips.

## Section 0 — Shared config & GPU-aware helpers
(Equivalent of `common.py`, inlined so the notebook is self-contained.)

In [2]:
from __future__ import annotations
import os, re

# Guard against the Windows 'multiple OpenMP runtimes' abort (torch + MKL).
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

# ---- Paths (relative to this notebook's folder) ----
ROOT_DIR    = os.path.abspath(os.getcwd())
RAW_CSV     = os.path.join(ROOT_DIR, "Dataset_5971.csv")
OUTPUTS_DIR = os.path.join(ROOT_DIR, "outputs")
TRAIN_TSV   = os.path.join(OUTPUTS_DIR, "train_nemo_format.tsv")
DEV_TSV     = os.path.join(OUTPUTS_DIR, "dev_nemo_format.tsv")
TEST_TSV    = os.path.join(OUTPUTS_DIR, "test_nemo_format.tsv")

# ---- Label convention: 0 = ham/legit, 1 = scam/phishing (smishing+spam) ----
LABEL_MAP   = {"ham": 0, "smishing": 1, "spam": 1}
LABEL_NAMES = {0: "ham", 1: "scam"}

def ensure_outputs_dir():
    os.makedirs(OUTPUTS_DIR, exist_ok=True)
    return OUTPUTS_DIR

def get_dataframe_lib():
    """Return (dataframe_module, is_gpu). cuDF first, pandas fallback."""
    try:
        import cudf
        print("[common] Using cuDF (GPU dataframe backend).")
        return cudf, True
    except Exception as exc:
        import pandas as pd
        print(f"[common] cuDF unavailable ({exc.__class__.__name__}); falling back to pandas (CPU).")
        return pd, False

def normalize_label(raw_label):
    """Map a raw label to a binary id; anything not 'ham' is scam (1)."""
    key = str(raw_label).strip().lower()
    return LABEL_MAP.get(key, 1 if key != "ham" else 0)

# Pre-compiled cleaning regexes.
_URL_RE  = re.compile(r"http[s]?://\S+|www\.\S+|\b\S+\.(?:com|net|org|io|co)/\S*", re.I)
_WS_RE   = re.compile(r"\s+")
_KEEP_RE = re.compile(r"[^a-zA-Z0-9!$%./\s]")

def clean_text(text, lowercase=False):
    """Normalise one SMS: strip mojibake, collapse URLs, drop exotic punctuation."""
    if text is None:
        return ""
    s = str(text)
    s = s.replace("\ufffd", " ")      # strip UTF replacement chars
    s = _URL_RE.sub(" URL ", s)       # collapse links to a URL token
    s = _KEEP_RE.sub(" ", s)          # drop exotic punctuation
    s = _WS_RE.sub(" ", s).strip()    # squeeze whitespace
    return s.lower() if lowercase else s

ensure_outputs_dir()
print("ROOT_DIR =", ROOT_DIR)


# ---- Deployment artifacts (for Streamlit Cloud) ----
ARTIFACTS_DIR = os.path.join(ROOT_DIR, "artifacts")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
# Shared metrics registry filled by Module 2 (baseline) and Module 3 (BERT),
# then written to artifacts/model_benchmark.csv.
BENCHMARK = {}
print("ARTIFACTS_DIR =", ARTIFACTS_DIR)

ROOT_DIR = /content/drive/MyDrive/MiniProject
ARTIFACTS_DIR = /content/drive/MyDrive/MiniProject/artifacts


## Module 1 — GPU Data Preprocessing
Load → clean → label → sequence-length analysis → write NeMo TSV splits `[CLEAN_TEXT]\t[LABEL_ID]` (no header, 70/15/15 stratified).

In [3]:
import pandas as pd

def load_raw(df_lib, is_gpu):
    try:
        df = df_lib.read_csv(RAW_CSV, encoding="latin-1")
    except TypeError:
        df = df_lib.read_csv(RAW_CSV)
    df = df[["TEXT", "LABEL"]]
    print(f"[01] Loaded {len(df)} rows ({'GPU/cuDF' if is_gpu else 'CPU/pandas'}).")
    return df

def clean_and_label(df, is_gpu):
    df = df.dropna(subset=["TEXT", "LABEL"])
    pdf = df.to_pandas() if is_gpu else df.copy()
    pdf["label_id"]   = pdf["LABEL"].map(normalize_label).astype(int)
    pdf["clean_text"] = pdf["TEXT"].map(lambda t: clean_text(t, lowercase=False))
    pdf = pdf[pdf["clean_text"].str.len() > 0].reset_index(drop=True)
    dist = pdf["label_id"].value_counts().to_dict()
    print(f"[01] Label distribution: ham(0)={dist.get(0,0)}  scam(1)={dist.get(1,0)}")
    return pdf

def sequence_length_analysis(pdf):
    lengths = pdf["clean_text"].str.split().map(len)
    stats = {"count": int(lengths.count()), "mean": round(float(lengths.mean()),2),
             "std": round(float(lengths.std()),2), "min": int(lengths.min()),
             "p50": int(lengths.quantile(.50)), "p90": int(lengths.quantile(.90)),
             "p95": int(lengths.quantile(.95)), "p99": int(lengths.quantile(.99)),
             "max": int(lengths.max())}
    print("[01] Sequence-length (word count) distribution:")
    for k, v in stats.items():
        print(f"       {k:>5}: {v}")
    return stats

def stratified_split(pdf, train_frac=0.7, dev_frac=0.15, seed=42):
    train_parts, dev_parts, test_parts = [], [], []
    for _, grp in pdf.groupby("label_id"):
        grp = grp.sample(frac=1.0, random_state=seed)
        n = len(grp); n_train = int(n*train_frac); n_dev = int(n*dev_frac)
        train_parts.append(grp.iloc[:n_train])
        dev_parts.append(grp.iloc[n_train:n_train+n_dev])
        test_parts.append(grp.iloc[n_train+n_dev:])
    comb = lambda parts: pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    train, dev, test = comb(train_parts), comb(dev_parts), comb(test_parts)
    print(f"[01] Split sizes -> train={len(train)}  dev={len(dev)}  test={len(test)}")
    return train, dev, test

def write_nemo_tsv(frame, path):
    with open(path, "w", encoding="utf-8") as fh:
        for text, label in zip(frame["clean_text"], frame["label_id"]):
            text = str(text).replace("\t", " ").replace("\n", " ").strip()
            fh.write(f"{text}\t{int(label)}\n")
    print(f"[01] Wrote {len(frame)} rows -> {os.path.relpath(path, ROOT_DIR)}")

df_lib, is_gpu = get_dataframe_lib()
df  = load_raw(df_lib, is_gpu)
pdf = clean_and_label(df, is_gpu)
sequence_length_analysis(pdf)
train_df, dev_df, test_df = stratified_split(pdf)
write_nemo_tsv(train_df, TRAIN_TSV)
write_nemo_tsv(dev_df,   DEV_TSV)
write_nemo_tsv(test_df,  TEST_TSV)
print("[01] Data preparation complete.")

[common] Using cuDF (GPU dataframe backend).
[01] Loaded 5971 rows (GPU/cuDF).
[01] Label distribution: ham(0)=4842  scam(1)=1127
[01] Sequence-length (word count) distribution:
       count: 5969
        mean: 16.28
         std: 11.23
         min: 1
         p50: 13
         p90: 29
         p95: 33
         p99: 53
         max: 173
[01] Split sizes -> train=4177  dev=895  test=897
[01] Wrote 4177 rows -> outputs/train_nemo_format.tsv
[01] Wrote 895 rows -> outputs/dev_nemo_format.tsv
[01] Wrote 897 rows -> outputs/test_nemo_format.tsv
[01] Data preparation complete.


## Module 2 — cuML Baseline (TF-IDF + LogisticRegression)
cuML GPU path with a scikit-learn CPU fallback. Reports Precision / Recall / F1 and inference runtime.

In [4]:
import time

def get_ml_backend():
    try:
        from cuml.feature_extraction.text import TfidfVectorizer
        from cuml.linear_model import LogisticRegression
        print("[02] Using cuML (GPU) TF-IDF + LogisticRegression.")
        return TfidfVectorizer, LogisticRegression, True
    except Exception as exc:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.linear_model import LogisticRegression
        print(f"[02] cuML unavailable ({exc.__class__.__name__}); falling back to scikit-learn (CPU).")
        return TfidfVectorizer, LogisticRegression, False

def load_tsv(df_lib, path):
    df = df_lib.read_csv(path, sep="\t", header=None, names=["text", "label"], quoting=3)
    return df.dropna()

def _host(a):
    import numpy as np
    if hasattr(a, "to_numpy"): return a.to_numpy()
    if hasattr(a, "get"):      return a.get()
    return np.asarray(a)

def evaluate(y_true, y_pred):
    import numpy as np
    from sklearn.metrics import classification_report, precision_recall_fscore_support
    yt, yp = _host(y_true).astype(int), _host(y_pred).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(yt, yp, average="binary", pos_label=1, zero_division=0)
    print("\n[02] === Test-set metrics (positive class = scam/1) ===")
    print(f"       Precision: {p:.4f}\n       Recall   : {r:.4f}\n       F1-Score : {f1:.4f}")
    print("\n[02] Full classification report:")
    print(classification_report(yt, yp, target_names=["ham", "scam"], zero_division=0))
    return p, r, f1

df_lib, df_is_gpu = get_dataframe_lib()
TfidfVectorizer, LogisticRegression, ml_is_gpu = get_ml_backend()
use_gpu = ml_is_gpu and df_is_gpu

train = load_tsv(df_lib, TRAIN_TSV)
test  = load_tsv(df_lib, TEST_TSV)
print(f"[02] Loaded train={len(train)}  test={len(test)}")

Xtr_text = train["text"].astype("str") if use_gpu else train["text"].astype(str).tolist()
Xte_text = test["text"].astype("str")  if use_gpu else test["text"].astype(str).tolist()
ytr = train["label"].astype("int32") if use_gpu else train["label"].astype(int).to_numpy()
yte = test["label"].astype("int32")  if use_gpu else test["label"].astype(int).to_numpy()

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000, sublinear_tf=True)
t0 = time.perf_counter(); Xtr = vec.fit_transform(Xtr_text); Xte = vec.transform(Xte_text)
t_feat = time.perf_counter() - t0
print(f"[02] TF-IDF fit+transform: {t_feat:.3f}s (features={Xtr.shape[1]})")

clf = LogisticRegression(max_iter=1000)
t0 = time.perf_counter(); clf.fit(Xtr, ytr); t_train = time.perf_counter() - t0
print(f"[02] Classifier train time: {t_train:.3f}s")

t0 = time.perf_counter(); ypred = clf.predict(Xte); t_infer = time.perf_counter() - t0
print(f"[02] Inference on {len(test)} rows: {t_infer:.4f}s ({len(test)/max(t_infer,1e-9):.0f} msg/s)")

evaluate(yte, ypred)
print(f"\n[02] Baseline summary -> backend={'GPU/cuML' if use_gpu else 'CPU/sklearn'}  "
      f"feat={t_feat:.3f}s  train={t_train:.3f}s  infer={t_infer:.4f}s")


# ---- Capture metrics + export baseline artifact ----
import numpy as _np
from sklearn.metrics import accuracy_score as _acc
_p, _r, _f1 = evaluate(yte, ypred)          # evaluate() returns (precision, recall, f1)
_yte_h = yte.to_numpy() if hasattr(yte, "to_numpy") else _np.asarray(yte)
_yp_h  = ypred.to_numpy() if hasattr(ypred, "to_numpy") else (
         ypred.get() if hasattr(ypred, "get") else _np.asarray(ypred))
BENCHMARK["Baseline (TF-IDF + LogReg)"] = {
    "precision": float(_p), "recall": float(_r), "f1": float(_f1),
    "accuracy": float(_acc(_yte_h.astype(int), _yp_h.astype(int))),
    "backend": "GPU/cuML" if use_gpu else "CPU/sklearn",
}
import joblib
_baseline_path = os.path.join(ARTIFACTS_DIR, "cuml_baseline.joblib")
joblib.dump({"vectorizer": vec, "model": clf}, _baseline_path)
print(f"[02] Saved baseline artifact -> {os.path.relpath(_baseline_path, ROOT_DIR)}")

[common] Using cuDF (GPU dataframe backend).
[02] Using cuML (GPU) TF-IDF + LogisticRegression.
[02] Loaded train=4177  test=897
[02] TF-IDF fit+transform: 31.318s (features=11179)
[02] Classifier train time: 10.737s
[02] Inference on 897 rows: 12.8466s (70 msg/s)

[02] === Test-set metrics (positive class = scam/1) ===
       Precision: 1.0000
       Recall   : 0.7647
       F1-Score : 0.8667

[02] Full classification report:
              precision    recall  f1-score   support

         ham       0.95      1.00      0.97       727
        scam       1.00      0.76      0.87       170

    accuracy                           0.96       897
   macro avg       0.97      0.88      0.92       897
weighted avg       0.96      0.96      0.95       897


[02] Baseline summary -> backend=GPU/cuML  feat=31.318s  train=10.737s  infer=12.8466s

[02] === Test-set metrics (positive class = scam/1) ===
       Precision: 1.0000
       Recall   : 0.7647
       F1-Score : 0.8667

[02] Full classificat

## Module 3 — BERT Fine-Tuning & Comparative Analysis (HuggingFace, GPU)

Fine-tunes BERT for scam detection with HuggingFace `transformers` (`Trainer` +
`AutoModelForSequenceClassification`). Chosen over NeMo for Colab reliability —
it matches Colab's PyTorch and installs in seconds — while producing the same
deployable artifact (`artifacts/bert_phishing_model/` with `config.json`,
`vocab.txt`, `model.safetensors`).

Two comparative loops (as required):
1. **Cased vs uncased** — `bert-base-cased` vs `bert-base-uncased`, to show
   capitalisation ("URGENT", "FREE", "CLAIM") carries scam signal.
2. **max_seq_length sweep** — 64 / 128 / 256 (memory vs accuracy).

Adam optimizer + linear warmup schedule, FP16 mixed precision (`fp16=True` on
GPU). Metrics log to TensorBoard under `outputs/hf_experiments/`; the best model
runs an error analysis to `outputs/error_analysis.txt` and exports HF weights.

Install once in Colab (then re-run this cell):
```
!pip install -q transformers datasets accelerate
```
On CPU it trains very slowly, so it auto-skips unless a GPU is present.

In [5]:
# ---- Module 3: BERT fine-tuning with HuggingFace transformers ----
def hf_available():
    """True only if transformers imports AND a CUDA GPU is visible."""
    try:
        import torch, transformers  # noqa: F401
        if not torch.cuda.is_available():
            print("[03] transformers present but no CUDA GPU -> skipping BERT.")
            return False
        return True
    except Exception as exc:
        print(f"[03] transformers not importable ({exc.__class__.__name__}: {exc}).")
        return False


def print_setup_help():
    print("\n[03] BERT fine-tuning needs a GPU runtime + HuggingFace transformers.")
    print("[03]   1. Colab: Runtime -> Change runtime type -> GPU.")
    print("[03]   2. !pip install -q transformers datasets accelerate")
    print("[03]   3. Re-run this cell.")


def _read_tsv(path):
    """Read a NeMo-format TSV (text\\tlabel, no header) -> (texts, labels)."""
    texts, labels = [], []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 2:
                texts.append(parts[0]); labels.append(int(parts[1]))
    return texts, labels


def _metrics_from_preds(labels, preds):
    from sklearn.metrics import precision_recall_fscore_support, accuracy_score
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", pos_label=1, zero_division=0)
    return {"precision": float(p), "recall": float(r), "f1": float(f1),
            "accuracy": float(accuracy_score(labels, preds))}


def run_experiment(pretrained_model, max_seq_length, epochs=3):
    """Fine-tune one (backbone, seq_len) config. Returns (trainer, tokenizer, metrics)."""
    import numpy as np
    import torch
    from datasets import Dataset
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              TrainingArguments, Trainer)

    print(f"\n[03] >>> backbone={pretrained_model}  max_seq_length={max_seq_length}")
    tok = AutoTokenizer.from_pretrained(pretrained_model)

    tr_txt, tr_lab = _read_tsv(TRAIN_TSV)
    dv_txt, dv_lab = _read_tsv(DEV_TSV)
    te_txt, te_lab = _read_tsv(TEST_TSV)

    def _encode(batch):
        return tok(batch["text"], truncation=True, padding="max_length",
                   max_length=max_seq_length)

    ds_tr = Dataset.from_dict({"text": tr_txt, "label": tr_lab}).map(_encode, batched=True)
    ds_dv = Dataset.from_dict({"text": dv_txt, "label": dv_lab}).map(_encode, batched=True)
    ds_te = Dataset.from_dict({"text": te_txt, "label": te_lab}).map(_encode, batched=True)
    cols = ["input_ids", "attention_mask", "label"]
    for d in (ds_tr, ds_dv, ds_te):
        d.set_format("torch", columns=[c for c in cols if c in d.column_names])

    model = AutoModelForSequenceClassification.from_pretrained(
        pretrained_model, num_labels=2)

    run_name = f"{pretrained_model}_seq{max_seq_length}"
    run_dir = os.path.join(OUTPUTS_DIR, "hf_experiments", run_name)

    # Build TrainingArguments defensively: the accepted kwargs differ across
    # transformers versions (e.g. 'warmup_ratio'/'logging_dir' were removed in
    # 5.x, and 'evaluation_strategy' was renamed to 'eval_strategy' in 4.46).
    # We only pass kwargs the installed version actually accepts.
    import inspect
    _accepted = set(inspect.signature(TrainingArguments.__init__).parameters)

    def _pick(**kw):
        return {k: v for k, v in kw.items() if k in _accepted}

    ta_kwargs = _pick(
        output_dir=run_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        learning_rate=2e-5,                 # Adam (HF default optimizer) ...
        weight_decay=0.01,
        warmup_ratio=0.1,                   # ... linear warmup (if supported)
        fp16=torch.cuda.is_available(),     # FP16 mixed precision on GPU
        logging_strategy="epoch",
        save_strategy="no",
        report_to=["tensorboard"],          # -> outputs/hf_experiments/<run>
        logging_dir=os.path.join(run_dir, "tb"),
        disable_tqdm=False,
    )
    # eval-every-epoch: use whichever arg name this version exposes.
    if "eval_strategy" in _accepted:
        ta_kwargs["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in _accepted:
        ta_kwargs["evaluation_strategy"] = "epoch"
    args = TrainingArguments(**ta_kwargs)

    def _compute(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return _metrics_from_preds(labels, preds)

    trainer = Trainer(model=model, args=args, train_dataset=ds_tr,
                      eval_dataset=ds_dv, compute_metrics=_compute)
    trainer.train()

    # Final metrics on the held-out TEST split.
    pred_out = trainer.predict(ds_te)
    preds = np.argmax(pred_out.predictions, axis=-1)
    metrics = _metrics_from_preds(np.asarray(te_lab), preds)
    print(f"[03] <<< {run_name}: {metrics}")
    return trainer, tok, metrics


def loop_cased_vs_uncased():
    """Loop 1: fix seq=128, vary backbone casing."""
    results = {}
    for backbone in ("bert-base-cased", "bert-base-uncased"):
        _, _, m = run_experiment(backbone, max_seq_length=128)
        results[backbone] = m
    print("\n[03] === Loop 1: Cased vs Uncased (seq=128) ===")
    for k, v in results.items():
        print(f"       {k:<20} -> {v}")
    return results


def loop_seq_length():
    """Loop 2: fix cased backbone, vary max_seq_length."""
    results = {}
    for msl in (64, 128, 256):
        _, _, m = run_experiment("bert-base-cased", max_seq_length=msl)
        results[msl] = m
    print("\n[03] === Loop 2: Sequence-length impact (bert-base-cased) ===")
    for k, v in results.items():
        print(f"       max_seq_length={k:<4} -> {v}")
    return results


def error_analysis(trainer, tokenizer, max_seq_length=128, top_k=15):
    """Dump the most confident FP / FN test mistakes to error_analysis.txt."""
    import numpy as np
    from datasets import Dataset
    te_txt, te_lab = _read_tsv(TEST_TSV)
    labels = np.asarray(te_lab)

    def _encode(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length",
                         max_length=max_seq_length)
    ds = Dataset.from_dict({"text": te_txt, "label": te_lab}).map(_encode, batched=True)
    ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    logits = trainer.predict(ds).predictions
    preds = np.argmax(logits, axis=-1)

    out_path = os.path.join(OUTPUTS_DIR, "error_analysis.txt")
    fp_idx = np.where((preds == 1) & (labels == 0))[0]
    fn_idx = np.where((preds == 0) & (labels == 1))[0]
    with open(out_path, "w", encoding="utf-8") as fh:
        fh.write("BERT (HuggingFace) - Test-set Error Analysis\n" + "=" * 60 + "\n")
        fh.write(f"Total test rows : {len(te_txt)}\n")
        fh.write(f"False Positives : {len(fp_idx)} (ham mis-flagged as scam)\n")
        fh.write(f"False Negatives : {len(fn_idx)} (scam missed as ham)\n\n")
        fh.write(f"--- Top {top_k} False Positives ---\n")
        for i in fp_idx[:top_k]: fh.write(f"[true=ham pred=scam] {te_txt[i]}\n")
        fh.write(f"\n--- Top {top_k} False Negatives ---\n")
        for i in fn_idx[:top_k]: fh.write(f"[true=scam pred=ham] {te_txt[i]}\n")
    print(f"[03] Error analysis written -> {os.path.relpath(out_path, ROOT_DIR)}")


def save_bert_artifacts(trainer, tokenizer, hf_dir="artifacts/bert_phishing_model"):
    """Save HF model + tokenizer so the folder has config.json, vocab.txt, weights."""
    hf_dir = hf_dir if os.path.isabs(hf_dir) else os.path.join(ROOT_DIR, hf_dir)
    os.makedirs(hf_dir, exist_ok=True)
    trainer.save_model(hf_dir)          # config.json + model.safetensors
    tokenizer.save_pretrained(hf_dir)   # vocab.txt + tokenizer config
    print(f"[03] Saved BERT HF artifacts -> {os.path.relpath(hf_dir, ROOT_DIR)}")
    return hf_dir


# ---- Run Module 3 ----
if hf_available():
    loop_cased_vs_uncased()            # comparison loop 1
    loop_seq_length()                  # comparison loop 2
    # Train the best overall config once for export + error analysis.
    best_trainer, best_tok, best_metrics = run_experiment(
        "bert-base-cased", max_seq_length=128)
    BENCHMARK["BERT (HF fine-tuned)"] = {**best_metrics, "backend": "GPU/transformers"}
    error_analysis(best_trainer, best_tok, max_seq_length=128)
    save_bert_artifacts(best_trainer, best_tok)
    print("[03] Done. TensorBoard: tensorboard --logdir outputs/hf_experiments")
else:
    print_setup_help()
    print("[03] BERT not trained -> skipping HF weight export.")



[03] >>> backbone=bert-base-cased  max_seq_length=128


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Map:   0%|          | 0/4177 [00:00<?, ? examples/s]

... (49 outputs left out)

Map:   0%|          | 0/897 [00:00<?, ? examples/s]

[03] Error analysis written -> outputs/error_analysis.txt


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[03] Saved BERT HF artifacts -> artifacts/bert_phishing_model
[03] Done. TensorBoard: tensorboard --logdir outputs/hf_experiments


## Export — Benchmark summary table
Writes Precision / Recall / F1 / Accuracy for every model that ran to `artifacts/model_benchmark.csv`.

In [6]:
import pandas as pd

# BENCHMARK is filled by Module 2 (always) and Module 3 (GPU only).
if not BENCHMARK:
    print("[bench] No metrics captured yet - run Modules 2/3 first.")
else:
    _rows = []
    for _name, _m in BENCHMARK.items():
        _rows.append({"model": _name,
                      "precision": round(_m["precision"], 4),
                      "recall": round(_m["recall"], 4),
                      "f1": round(_m["f1"], 4),
                      "accuracy": round(_m["accuracy"], 4),
                      "backend": _m.get("backend", "")})
    _bench_df = pd.DataFrame(_rows)[["model", "precision", "recall", "f1",
                                     "accuracy", "backend"]]
    _bench_path = os.path.join(ARTIFACTS_DIR, "model_benchmark.csv")
    _bench_df.to_csv(_bench_path, index=False)
    print(f"[bench] Wrote {len(_bench_df)} rows -> "
          f"{os.path.relpath(_bench_path, ROOT_DIR)}")
    print(_bench_df.to_string(index=False))

[bench] Wrote 2 rows -> artifacts/model_benchmark.csv
                     model  precision  recall     f1  accuracy          backend
Baseline (TF-IDF + LogReg)     1.0000  0.7647 0.8667    0.9554         GPU/cuML
      BERT (HF fine-tuned)     0.9877  0.9471 0.9670    0.9877 GPU/transformers


## Module 4 — Word Co-occurrence Graph Analytics
Scam-only corpus → tokenise (stop-word removal) → weighted co-occurrence graph → **PageRank** (top-20 hubs) + **Louvain** communities. cuGraph GPU native → NetworkX (optionally GPU via nx-cugraph) fallback.

In [7]:
from collections import Counter
from itertools import combinations

STOP_WORDS = set("""
a an and are as at be but by for from has have he her his i if in is it its
me my no not of on or our so that the their them then there they this to too
up us was we were what when where which who will with you your yours ok im
u ur 2 4 n r s t m re ve ll d
""".split())
TOKEN_RE = re.compile(r"[a-zA-Z][a-zA-Z0-9]{1,}")
TOP_PAGERANK = 20

def load_scam_messages():
    texts = []
    for path in (TRAIN_TSV, DEV_TSV, TEST_TSV):
        if not os.path.exists(path): continue
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                parts = line.rstrip("\n").split("\t")
                if len(parts) >= 2 and parts[1].strip() == "1":
                    texts.append(parts[0])
    print(f"[04] Loaded {len(texts)} scam messages.")
    return texts

def tokenize(text):
    return [t for t in (w.lower() for w in TOKEN_RE.findall(text)) if t not in STOP_WORDS]

def build_edges(texts, min_edge_weight=2):
    pair_counts = Counter()
    for text in texts:
        tokens = sorted(set(tokenize(text)))
        for a, b in combinations(tokens, 2):
            pair_counts[(a, b)] += 1
    srcs, dsts, weights = [], [], []
    for (a, b), w in pair_counts.items():
        if w >= min_edge_weight:
            srcs.append(a); dsts.append(b); weights.append(w)
    print(f"[04] Built {len(srcs)} edges (min_edge_weight={min_edge_weight}) over {len({*srcs, *dsts})} nodes.")
    return srcs, dsts, weights

def _group_communities(node2comm):
    grouped = {}
    for node, cid in node2comm.items():
        grouped.setdefault(cid, []).append(node)
    return grouped

def analyze_with_cugraph(srcs, dsts, weights):
    import cudf, cugraph
    gdf = cudf.DataFrame({"src": srcs, "dst": dsts, "weight": weights})
    G = cugraph.Graph()
    G.from_cudf_edgelist(gdf, source="src", destination="dst", edge_attr="weight", renumber=True)
    pr = cugraph.pagerank(G).sort_values("pagerank", ascending=False)
    top_pr = list(zip(pr["vertex"].to_pandas().tolist(), pr["pagerank"].to_pandas().tolist()))[:TOP_PAGERANK]
    parts, _ = cugraph.louvain(G); parts = parts.to_pandas()
    communities = _group_communities(dict(zip(parts["vertex"], parts["partition"])))
    return top_pr, communities

def analyze_with_networkx(srcs, dsts, weights):
    import networkx as nx
    G = nx.Graph()
    for s, d, w in zip(srcs, dsts, weights):
        G.add_edge(s, d, weight=w)
    backend = "nx-cugraph(GPU)" if os.environ.get("NX_CUGRAPH_AUTOCONFIG") else "networkx(CPU)"
    print(f"[04] Graph analytics backend: {backend}")
    pr = nx.pagerank(G, weight="weight")
    top_pr = sorted(pr.items(), key=lambda kv: kv[1], reverse=True)[:TOP_PAGERANK]
    try:
        comms = nx.community.louvain_communities(G, weight="weight", seed=42)
    except AttributeError:
        comms = nx.community.greedy_modularity_communities(G, weight="weight")
    node2comm = {}
    for cid, nodes in enumerate(comms):
        for nd in nodes: node2comm[nd] = cid
    return top_pr, _group_communities(node2comm)

def report(top_pr, communities, degree_lookup):
    lines = ["Scam-corpus Word Co-occurrence Graph Analytics", "="*60]
    lines.append(f"\nTop {TOP_PAGERANK} PageRank hub terms")
    lines.append(f"{'rank':>4}  {'term':<18}{'pagerank':>12}"); lines.append("-"*36)
    for i, (term, score) in enumerate(top_pr, 1):
        lines.append(f"{i:>4}  {str(term):<18}{score:>12.6f}")
    lines.append(f"\nLouvain vocabulary clusters: {len(communities)} communities"); lines.append("-"*60)
    ranked = sorted(communities.items(), key=lambda kv: len(kv[1]), reverse=True)
    for cid, nodes in ranked[:8]:
        top_terms = sorted(nodes, key=lambda n: degree_lookup.get(n, 0), reverse=True)[:12]
        lines.append(f"Cluster {cid:<3} (size={len(nodes):>4}): {', '.join(map(str, top_terms))}")
    text = "\n".join(lines); print("\n" + text)
    out_path = os.path.join(OUTPUTS_DIR, "graph_analysis.txt")
    with open(out_path, "w", encoding="utf-8") as fh: fh.write(text + "\n")
    print(f"\n[04] Report written -> {os.path.relpath(out_path, ROOT_DIR)}")

texts = load_scam_messages()
if texts:
    srcs, dsts, weights = build_edges(texts, min_edge_weight=2)
    if srcs:
        deg = Counter()
        for s, d, w in zip(srcs, dsts, weights):
            deg[s] += w; deg[d] += w
        try:
            top_pr, communities = analyze_with_cugraph(srcs, dsts, weights)
            print("[04] Graph analytics backend: cuGraph (GPU native).")
        except Exception as exc:
            print(f"[04] cuGraph unavailable ({exc.__class__.__name__}); using NetworkX path.")
            top_pr, communities = analyze_with_networkx(srcs, dsts, weights)
        report(top_pr, communities, deg)
        print("[04] Graph analytics complete.")
    else:
        print("[04] No edges survived the weight filter; lower min_edge_weight.")
else:
    print("[04] No scam messages found. Run Module 1 first.")


# ---- Export graph analytics for Streamlit Cloud ----
import json
if "top_pr" in dir() and "communities" in dir():
    # Rank clusters by size; name each by its highest-degree terms.
    _ranked = sorted(communities.items(), key=lambda kv: len(kv[1]), reverse=True)
    _clusters = []
    for _cid, _nodes in _ranked:
        _top_terms = sorted(_nodes, key=lambda n: deg.get(n, 0), reverse=True)[:15]
        _clusters.append({"cluster_id": int(_cid), "size": len(_nodes),
                          "top_terms": list(map(str, _top_terms))})
    _graph_payload = {
        "top_pagerank": [{"term": str(_t), "pagerank": float(_s)} for _t, _s in top_pr],
        "num_communities": len(communities),
        "louvain_clusters": _clusters,
    }
    _graph_path = os.path.join(ARTIFACTS_DIR, "graph_clusters.json")
    with open(_graph_path, "w", encoding="utf-8") as _fh:
        json.dump(_graph_payload, _fh, indent=2)
    print(f"[04] Saved graph analytics -> {os.path.relpath(_graph_path, ROOT_DIR)}")
else:
    print("[04] No graph results in scope - run the Module 4 cell above first.")

[04] Loaded 1127 scam messages.
[04] Built 18849 edges (min_edge_weight=2) over 1361 nodes.
[04] cuGraph unavailable (ModuleNotFoundError); using NetworkX path.
[04] Graph analytics backend: networkx(CPU)

Scam-corpus Word Co-occurrence Graph Analytics

Top 20 PageRank hub terms
rank  term                  pagerank
------------------------------------
   1  call                  0.032022
   2  now                   0.017843
   3  free                  0.015612
   4  url                   0.013011
   5  txt                   0.011754
   6  mobile                0.010275
   7  claim                 0.010004
   8  stop                  0.008979
   9  reply                 0.008604
  10  text                  0.008548
  11  customer              0.007981
  12  get                   0.007451
  13  prize                 0.007329
  14  only                  0.007064
  15  please                0.006809
  16  been                  0.006511
  17  just                  0.006186
  18  contact    